In [1]:
import numpy as np
import pandas as pd
import os

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'raw')
DEM_PATH = os.path.join(DATA_DIR, 'DEM.npz')
MASK_PATH = os.path.join(DATA_DIR, 'Mounds_raster_mask_opened_closed.npy')

In [2]:
with np.load(DEM_PATH) as data:
    start_dem = data['dataset'].astype(np.float32, copy=False)
    valid_mask = data['validMask'].astype(np.uint8, copy=False)
    mask = np.load(MASK_PATH)

In [3]:
vm_bool = valid_mask.astype(bool)

In [4]:
dem = start_dem.copy()
dem[~vm_bool] = np.nan

In [5]:
labels_combined = mask.astype(np.int32, copy=False).copy()
labels_combined[~vm_bool] = 255

In [6]:
print('dem:', dem.shape, dem.dtype)
print('labels_combined:', labels_combined.shape, labels_combined.dtype)
print('valid_mask:', valid_mask.shape, valid_mask.dtype)
print('unique(valid_mask):', np.unique(valid_mask))
print('sum(valid_mask == 1):', np.sum(valid_mask == 1))
print('sum(valid_mask != 0):', np.sum(valid_mask != 0))
print('sum(valid_mask == 0):', np.sum(valid_mask == 0))

dem: (17092, 9791) float32
labels_combined: (17092, 9791) int32
valid_mask: (17092, 9791) uint8
unique(valid_mask): [  0 255]
sum(valid_mask == 1): 0
sum(valid_mask != 0): 162275193
sum(valid_mask == 0): 5072579


In [7]:
IGNORE_INDEX = 255  # или ваш
# если labels_combined еще не создан:
labels_combined = mask.astype(np.int32, copy=False).copy()
labels_combined[~vm_bool] = IGNORE_INDEX

# Проверки
print('valid pixels:', vm_bool.sum())
print('unique classes (all):', np.unique(labels_combined))
print('unique classes (только валид):', np.unique(labels_combined[vm_bool]))

valid pixels: 162275193
unique classes (all): [  0   1 255]
unique classes (только валид): [0 1]


In [8]:
pos_percentage = labels_combined[vm_bool].sum()/vm_bool.sum()
print(f"{pos_percentage:6f}")

0.002785


In [14]:
# Generate all tile coordinates
H, W = dem.shape
coords = []
tile_size = 128
stride = 32
pos_only = True

for y in range(0, H - tile_size + 1, stride):
    for x in range(0, W - tile_size + 1, stride):
        if y + tile_size > H or x + tile_size > W:
            continue
        # Check if tile has any valid DEM data
        if vm_bool[y:y+tile_size, x:x+tile_size].any():
            if pos_only:
                if mask[y:y+tile_size, x:x+tile_size].any():
                    coords.append((y, x))
            else:
                coords.append((y, x))

coords = np.array(coords)

total_tiles_pos_percentage = []

for idx in range(len(coords)):
    y, x = coords[idx]
    dem_tile = dem[y:y+tile_size, x:x+tile_size]
    mask_tile = mask[y:y+tile_size, x:x+tile_size]
    valid_tile = vm_bool[y:y+tile_size, x:x+tile_size]

    pos_count = np.sum(mask_tile & valid_tile)
    tile_area = tile_size * tile_size
    
    tile_pos_percentage = pos_count / tile_area
    total_tiles_pos_percentage.append(tile_pos_percentage)

print(f"{np.mean(total_tiles_pos_percentage):6f}")

0.016575
